# 공통 Neo4j 적재와 실제 키 검증 (B06·B07)

`school_normalize_bundle.ipynb`가 만든 적재 패키지(`bundle.json`)를 **공통 Neo4j**에 넣는다.

1. 스키마: 온톨로지 v3.1 5-3절의 고유 제약 6개 + Chunk 전문 검색(cjk) + stage·scope 인덱스 + 벡터 인덱스(팀 합의 `text-embedding-3-small`, 1536차원, cosine). `IF NOT EXISTS`는 기존 인덱스 설정을 바꾸지 않으므로 실제 설정(분석기·차원)을 다시 읽어 확인한다.
2. 표준 사전 노드(Disease 11·SchoolRole 11·ActionConcept 27) MERGE.
3. 적재 → 담당 문서 범위의 실제 키·속성 대조 → **같은 입력 재적재** 후 키 집합 비교.
4. 끝점 누락(없는 청크로 가는 관계)이 조용히 사라지지 않고 트랜잭션 전체가 되돌려지는지 확인.
5. Chunk 임베딩(바뀐 청크만) → 6. 대표 질문 경로(청크 → 행동 → 역할) 읽기 조회.

모든 쓰기는 MERGE·IF NOT EXISTS이며 삭제·변경 문장은 없다. 접속 정보는 `.env`에서만 읽고 출력하지 않는다. 결과는 `data/interim/school/load_ledger.jsonl`에 남는다.

현재 적용본은 `data/specs/school_active_dataset.json`에서 명시적으로 읽는다. 로컬 패키지가 없으면 과거 경로로 대체하지 않는다. 서비스 데이터 교체·복구는 `apply_reviewed_migration.py`를 사용하며, 이 노트북의 스키마/사전 초기화·재적재 셀은 별도 작업이다. 11/06에는 입력 연결과 패키지만 검증했고 DB 쓰기 셀은 재실행하지 않았다.


In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from index.school.dictionaries import load_dictionaries
from index.school.embed import EMBEDDING_MODEL, embed_chunks, make_embedder
from index.school.load import (
    LoadError,
    fetch_scope_keys,
    is_clean,
    load_bundle,
    load_with_retry,
    verify_loaded_bundle,
)
from index.school.neo4j_local import get_driver
from index.school.schema_init import apply_schema, missing_dictionary_refs, schema_state, seed_dictionaries

ACTIVE = json.loads((ROOT / "data/specs/school_active_dataset.json").read_text(encoding="utf-8"))
if ACTIVE["status"] != "applied":
    raise RuntimeError("현재 적용본 상태가 아닙니다. 이행 기록과 패키지를 먼저 확인하세요.")
BUNDLE_PATH = ROOT / ACTIVE["bundle_path"]
BUNDLE_DIR = BUNDLE_PATH.parent
LEDGER = ROOT / "data" / "interim" / "school" / "load_ledger.jsonl"
bundle = json.loads(BUNDLE_PATH.read_text(encoding="utf-8"))
bundle["dictionary_refs"] = [tuple(x) for x in bundle["dictionary_refs"]]
print("패키지:", BUNDLE_DIR.name, {k: len(v) for k, v in bundle["nodes"].items()}, "관계", len(bundle["edges"]))

driver, DATABASE = get_driver(ROOT / ".env")
print("Neo4j 연결 확인")

## 1. 스키마(제약·인덱스)

In [ ]:
before = schema_state(driver, DATABASE)
print("적용 전 빠진 고유 제약:", before["missing_constraints"])
apply_schema(driver, DATABASE)
after = schema_state(driver, DATABASE)
print("적용 후 빠진 고유 제약:", after["missing_constraints"] or "없음",
      "/ 빠진 인덱스:", after["missing_indexes"] or "없음", "/ 설정이 다른 인덱스:", after["wrong_index_options"] or "없음")
display(pd.DataFrame([{"index": k, "type": v[0], "state": v[1]} for k, v in sorted(after["indexes"].items())]))
assert not after["missing_constraints"] and not after["missing_indexes"] and not after["wrong_index_options"]

## 2. 표준 사전 노드

In [ ]:
books = load_dictionaries()
print("사전 MERGE:", seed_dictionaries(driver, books, DATABASE))
missing = missing_dictionary_refs(driver, bundle["dictionary_refs"], DATABASE)
print("패키지가 연결할 사전 노드 중 DB에 없는 것:", missing or "없음")
assert not missing

## 3. 적재와 실제 키 검증

적재 카운터가 아니라 DB를 다시 읽어 기대 키·속성과 비교한다. 비교 범위는 school 담당 문서 2개와 그 ID로 시작하는 청크·행동, 거기서 나가는 관계다(팀원 데이터 건수와 섞지 않음).

In [ ]:
first = load_with_retry(driver, bundle, ledger=LEDGER, bundle_dir=BUNDLE_DIR.name, database=DATABASE)
print("상태:", first["status"])
print("처리 행 수:", first["written"])
print({k: (len(v) if isinstance(v, list) else v) for k, v in first["report"].items()})
assert is_clean(first["report"])
keys_after_first = fetch_scope_keys(driver, bundle, DATABASE)

## 4. 같은 입력 재적재 - 키 집합이 그대로인지

In [ ]:
second = load_with_retry(driver, bundle, ledger=LEDGER, bundle_dir=BUNDLE_DIR.name, database=DATABASE)
keys_after_second = fetch_scope_keys(driver, bundle, DATABASE)
same_nodes = keys_after_first["nodes"] == keys_after_second["nodes"]
same_edges = keys_after_first["edges"] == keys_after_second["edges"]
print("재적재 상태:", second["status"], "/ 노드 키·속성 동일:", same_nodes, "/ 관계 키 동일:", same_edges)
print("관계 수:", len(keys_after_second["edges"]), "(중복 관계가 생기면 늘어난다)")
assert same_nodes and same_edges and is_clean(second["report"])

## 5. 끝점 누락 검출 - 조용히 사라지지 않고 전체가 되돌려지는지

없는 청크(`…p0036:t09r09`)로 가는 `SUPPORTED_BY`를 하나 섞은 패키지를 넣어 본다. MATCH가 그 행을 버리면 처리 행 수가 줄어 `LoadError`가 나고, 행동 트랜잭션 전체가 되돌려져야 한다. 이후 검증이 여전히 깨끗해야 한다.

In [ ]:
broken = json.loads(json.dumps(bundle))
action_id = broken["nodes"]["SchoolAction"][0]["action_id"]
broken["edges"].append({"type": "SUPPORTED_BY", "start_label": "SchoolAction", "start_id": action_id,
                        "end_label": "Chunk", "end_id": "moe_school_manual_v3:p0036:t09r09", "props": {}})
try:
    load_bundle(driver, broken, DATABASE)
    raise AssertionError("끝점 누락이 검출되지 않음")
except LoadError as exc:
    print("검출:", exc)
report = verify_loaded_bundle(driver, bundle, DATABASE)
print("되돌린 뒤 검증 깨끗함:", is_clean(report))
assert is_clean(report)

## 6. Chunk 임베딩

Chunk.text를 팀 합의 모델로 임베딩해 `embedding`에 넣는다. 텍스트·모델이 장부(`embed_ledger.jsonl`)와 같고 DB에 벡터가 있으면 건너뛴다(재실행 비용 없음).

In [ ]:
embedded = embed_chunks(driver, bundle["nodes"]["Chunk"], make_embedder(),
                        ledger=ROOT / "data" / "interim" / "school" / "embed_ledger.jsonl", database=DATABASE)
print("모델:", EMBEDDING_MODEL, "/ 새로 임베딩:", len(embedded["embedded"]), "/ 건너뜀:", len(embedded["skipped"]))
print("임베딩 후 검증 깨끗함:", is_clean(verify_loaded_bundle(driver, bundle, DATABASE)))

## 7. 대표 질문 경로 미리 보기 (읽기 전용)

'유증상 학생 발견 시 담임·보건교사가 확인할 일'(S-Q1)에 쓰일 경로: 대응 1단계 행동 → 근거 청크 → 문서, 수행 역할과 basis. 실제 검색기(B08)는 Chunk 검색을 먼저 하지만, 여기서는 적재된 경로가 이어지는지만 본다.

In [ ]:
records, _, _ = driver.execute_query('''
MATCH (d:Document)-[:HAS_CHUNK]->(c:Chunk)<-[:SUPPORTED_BY]-(a:SchoolAction)-[r:DONE_BY]->(role:SchoolRole)
WHERE d.document_id IN $docs AND a.stage = 'RESPONSE_1' AND role.role_id IN ['HOMEROOM_TEACHER', 'HEALTH_TEACHER']
OPTIONAL MATCH (a)-[:INSTANCE_OF]->(k:ActionConcept)
RETURN d.document_id AS document, c.pdf_page AS pdf, c.printed_page AS printed, role.role_id AS role,
       r.basis AS basis, a.step_order AS step, a.action_id AS action_id, a.action_text AS action_text,
       k.concept_id AS concept
ORDER BY document, role, step, action_id''', docs=[d["document_id"] for d in bundle["nodes"]["Document"]],
    database_=DATABASE)
paths = pd.DataFrame([r.data() for r in records])
print("경로 수:", len(paths), "/ 문서·역할별:", dict(Counter(zip(paths["document"], paths["role"]))))
print("basis:", dict(Counter(paths["basis"])), "/ 개념 연결:", int(paths["concept"].notna().sum()), "/", len(paths))
pd.set_option("display.max_colwidth", 70)
display(paths)

In [ ]:
for line in LEDGER.read_text(encoding="utf-8").splitlines()[-2:]:
    entry = json.loads(line)
    print(entry["timestamp_utc"], entry["status"], entry["bundle_sha256"][:12], entry["verify"]["clean"])
driver.close()